# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each record set and field can be referenced by its `@id`. This is crucial for reproducible and standards-based data access.

In [ ]:
# List all available record sets, their @id, name, and contained fields
print("Available record sets:")
record_sets = meta.record_sets
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}, name: {rs.name}")
    print("  Fields:")
    for fld in rs.fields:
        print(f"    - Field @id: {fld.id}, name: {fld.name}, dataType: {fld.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Note:** For this example, we extract all main record sets. Update the IDs according to those found above.

In [ ]:
# Extract data from each record set using the record set `@id`

# Gather @id for all record sets
record_set_ids = [rs.id for rs in meta.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Use dataset.records(record_set=...) with @id
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for RecordSet {record_set_id}, shape: {dataframes[record_set_id].shape}")
    else:
        print(f"No records found for RecordSet {record_set_id}")

# Example: print columns and a preview for the first record set
if record_set_ids:
    main_rs_id = record_set_ids[0]
    if main_rs_id in dataframes:
        print(f"\nColumns for RecordSet @{main_rs_id}:")
        print(dataframes[main_rs_id].columns.tolist())
        display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes.

All fields/columns are referenced by their `@id`.

In [ ]:
# Select a numeric field for analysis (by @id), e.g., 'coverage' or 'Abundance_sample_1'
# Find a recordset and a numeric field for demonstration

import numpy as np

chosen_rs = None
numeric_field_id = None

# Search for the first numeric field
for rs in meta.record_sets:
    df = dataframes.get(rs.id)
    if df is not None:
        for fld in rs.fields:
            # Heuristic: Pick first float/integer field
            if fld.data_type and (fld.data_type.lower() in ['float', 'number', 'integer'] or 'abundance' in fld.id.lower()):
                if fld.id in df.columns:
                    chosen_rs = rs.id
                    numeric_field_id = fld.id
                    break
        if numeric_field_id:
            break

if numeric_field_id is None:
    print("No numeric field found for analysis.")
else:
    print(f"Using RecordSet: {chosen_rs}, Numeric Field: {numeric_field_id}")
    # Filter: remove NaN and apply a threshold (for demo)
    df = dataframes[chosen_rs]
    threshold = np.nanmean(df[numeric_field_id]) if np.issubdtype(df[numeric_field_id].dtype, np.number) else 0

    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {filtered_df.shape[0]} records.")
    display(filtered_df.head())

    # Normalize
    normed = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    filtered_df[numeric_field_id + '_normalized'] = normed
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # If a categorical/grouping field is present (e.g. 'description', 'accession'), group by it
    group_field = None
    for candidate in ['description', 'accession', 'gene_name', 'protein_name']:
        for fld in meta.record_set_by_id(chosen_rs).fields:
            if candidate in fld.id.lower():
                group_field = fld.id
                break
        if group_field:
            break

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
    else:
        print("No grouping field found for mean aggregation.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized counterpart.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and filtered_df.shape[0] > 0:
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(filtered_df[numeric_field_id], kde=True, ax=ax[0])
    ax[0].set_title(f"Distribution of {numeric_field_id}")
    sns.histplot(filtered_df[numeric_field_id + '_normalized'], kde=True, ax=ax[1], color='orange')
    ax[1].set_title(f"Normalized {numeric_field_id}")
    plt.tight_layout()
    plt.show()
else:
    print('No data available to plot.')

## 6. Conclusion
In this notebook, we've demonstrated how to load, inspect, and process the FAIR^2 annotated mass spectrometry dataset using `mlcroissant`. You can now proceed to more advanced analyses, modeling, or data integration tasks while referencing dataset entities using their stable `@id` identifiers.